# 08 · Final models

Active-sequence GRUs and LSTMs (5 epochs, several seeds), adding LSTM to the inference engine and building the final ensembles.

Needs `lib.py` and the `.npy` data written by `00_setup_data_feature_importance`. Comments are in Russian; outputs are kept as the record of each experiment.


In [1]:
from google.colab import drive
import os, time, shutil, subprocess, json
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/wnn/lib.py'), 'нет папки wnn: расшарь её со старого аккаунта и добавь ярлык в «Мой диск»'
%run -i /content/drive/MyDrive/wnn/lib.py
!pip -q install onnx
import torch, pyarrow.parquet as pq
from scipy.signal import lfilter
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
D = '/content/data'; os.makedirs(D, exist_ok=True)
sh = lambda c: print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip())
def ok(p, n):
    try: return pq.ParquetFile(p).num_row_groups == n
    except Exception: return False

# ---------- данные ----------
built = all(os.path.exists(f'{D}/{f}') for f in ('trainall_Y.npy', 'validrest_Y.npy', 'valid300_M.npy'))
if not built and not (ok(f'{ROOT}/train.parquet', 10607) and ok(f'{ROOT}/valid.parquet', 1873)):
    t = time.time(); print('качаю стартпак (10–15 минут)...', flush=True)
    sh("cd /content && curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | "
       "tar -xz --wildcards '*/baseline/*' '*/datasets/train.parquet' '*/datasets/valid.parquet' 2>/dev/null")
    print(f'скачано за {time.time() - t:.0f} с')
META = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
def build16(src, dst, rows):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet'); feat = [c for c in pf.schema_arrow.names if c not in META]; assert len(feat) == NF
    X = np.lib.format.open_memmap(f'{D}/{dst}_X16.npy', 'w+', np.float16, (len(rows), L, NF)); Y = np.zeros((len(rows), L, 2), np.float32); t = time.time()
    for k, i in enumerate(rows):
        tb = pf.read_row_group(int(i), columns=feat + ['t0', 't1'])
        X[k] = np.column_stack([tb[c].to_numpy() for c in feat]).astype(np.float16); Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if k % 2000 == 0: print(f'  {dst} {k}/{len(rows)} {time.time() - t:.0f} с', flush=True)
    X.flush(); del X; np.save(f'{D}/{dst}_Y.npy', Y)
held = set(np.random.default_rng(1).choice(1873, 300, replace=False).tolist())
if not os.path.exists(f'{D}/valid300_M.npy'): build('valid', 'valid300', 300, 1, True)
if not os.path.exists(f'{D}/validrest_Y.npy'): build16('valid', 'validrest', [i for i in range(1873) if i not in held])
if not os.path.exists(f'{D}/trainall_Y.npy'): build16('train', 'trainall', np.arange(10607))
for f in ('train', 'valid'):
    if os.path.exists(f'{ROOT}/{f}.parquet'): os.remove(f'{ROOT}/{f}.parquet')
sh('df -h /content | tail -1')

Xv = np.load(f'{D}/valid300_X.npy', mmap_mode='r'); Yv = np.load(f'{D}/valid300_Y.npy'); Mv = np.load(f'{D}/valid300_M.npy').astype(bool)
XA = np.load(f'{D}/trainall_X16.npy', mmap_mode='r'); YA = np.load(f'{D}/trainall_Y.npy', mmap_mode='r')
XR = np.load(f'{D}/validrest_X16.npy', mmap_mode='r'); YR = np.load(f'{D}/validrest_Y.npy')
assert len(XA) == 10607 and len(XR) == 1573
famv = fam_of(Yv, verbose=False); hv = np.bincount(famv, minlength=4) / len(famv); famR = fam_of(YR, verbose=False)
famT = np.concatenate([fam_of(np.asarray(YA[i:i + 500]), verbose=False) for i in range(0, len(YA), 500)])
Xva, Yva, Mva, fam_v, have = Xv, Yv, Mv, famv, hv

# ---------- predict из lib.py: float16 → float32 ----------
class F32:
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return np.asarray(s.X[i], np.float32)
if not getattr(predict, '_f32', False):
    _predict_lib = predict
    def predict(model, X, idx, *a, **k):
        if getattr(X, 'dtype', None) == np.float16: X = F32(X)
        return _predict_lib(model, X, idx, *a, **k)
    predict._f32 = True

# ---------- схема «train → valid»: признаки, модель, метрика, прогон ----------
N_TR, EP = 3000, 3
ids = np.sort(np.random.default_rng(0).choice(10607, N_TR, replace=False)); YTr = np.asarray(YA[ids]); famTr = famT[ids]
z_ = np.load(f'{CKPT_DIR}/evfeat_scale.npz'); MU8, SD8 = z_['MU'], z_['SD']
def evfeat_n(x, n=20000):
    m_ = len(x); d = np.diff(x[:, :22], axis=0, prepend=x[:1, :22]); ch = (d != 0).any(1); ch[0] = False
    m = np.where(ch, d.mean(1), 0.0); ev = np.flatnonzero(ch); cnt = np.cumsum(ch)
    cs = np.concatenate([[0.0], np.cumsum(m[ev])]); out = np.zeros((m_, 8), np.float32); t = np.arange(m_)
    for i, k in enumerate((1, 2, 3, 5, 10)): out[:, i] = cs[cnt] - cs[np.maximum(cnt - k, 0)]
    last = ev[np.maximum(cnt - 1, 0)] if len(ev) else np.zeros(m_, int)
    out[:, 5] = np.log1p(np.where(cnt > 0, t - last, n)); c0 = np.concatenate([[0], cnt])
    for i, w in ((6, 50), (7, 200)): out[:, i] = c0[t + 1] - c0[np.maximum(t + 1 - w, 0)]
    return out
def _ema(x, a=1 / 500): return lfilter([a], [1, a - 1], x - x[:1], axis=0) + x[:1]
V = np.r_[22:44, 74:96]
def t_volz(x):
    y = x.copy(); m = _ema(x[:, V]); v = _ema((x[:, V] - m) ** 2); y[:, V] = np.clip((x[:, V] - m) / np.sqrt(v + 1e-2), -10, 10); return y
FEAT = {'ev': (lambda x: ((evfeat_n(x) - MU8) / SD8).astype(np.float32), 8),
        'ema': (lambda x: np.clip(x - _ema(x), -10, 10).astype(np.float32), 112), '!volz': (None, 0)}
TRANS = {'volz': t_volz}
def getx(X, i, feats):
    b = np.asarray(X[np.atleast_1d(i)], np.float32)
    for f in feats:
        if f.startswith('!'): b = np.stack([TRANS[f[1:]](v) for v in b]).astype(np.float32)
    add = [np.stack([FEAT[f][0](v) for v in b]) for f in feats if not f.startswith('!')]
    return np.concatenate([b] + add, -1) if add else b
class G(nn.Module):
    def __init__(s, nin, H, drop): super().__init__(); s.gru = nn.GRU(nin, H, 2, batch_first=True, dropout=drop); s.head = nn.Linear(H, 2)
    def forward(s, x, h=None): y, h = s.gru(x, h); return s.head(y), h
def wpear(p, y, w):
    sw = w.sum() + 1e-8; my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    return ((w * (y - my) * (p - mp)).sum() / sw) / torch.sqrt(((w * (y - my) ** 2).sum() / sw) * ((w * (p - mp) ** 2).sum() / sw) + 1e-12)
@torch.no_grad()
def pred(net, X, ids_, feats, bs=50):
    net.eval(); out = []
    for s in range(0, len(ids_), bs):
        x = torch.from_numpy(getx(X, ids_[s:s + bs], feats)).to(dev); h = None; o_ = []
        for t in range(0, L, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)
def sums(P, Y, M):
    T = np.zeros((2, 6))
    for j in (0, 1):
        y = np.clip(Y[..., j][M], -2, 2).astype(np.float64); p = P[..., j][M].astype(np.float64); a = np.abs(y)
        T[j] = [a.sum(), (a * y).sum(), (a * p).sum(), (a * y * y).sum(), (a * p * p).sum(), (a * y * p).sum()]
    return T
def corrT(T):
    sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
    return float(np.mean((T[:, 5] / sw - my * mp) / np.sqrt((T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2))))
post = np.arange(L) >= WARM
RJ = f'{CKPT_DIR}/results_train_to_valid.json'; RES2 = json.load(open(RJ)) if os.path.exists(RJ) else {}
def run2(name, H=128, loss='wmse', feats=(), drop=0.0, wd=0.0, lr=2e-3, reweight=False, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); nin = 112 + sum(FEAT[f][1] for f in feats)
    net = G(nin, H, drop).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd) if wd > 0 else torch.optim.Adam(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EP * (N_TR // 32) * (L // 1000), pct_start=0.05)
    trmix = np.bincount(famTr, minlength=4) / N_TR; wf = VALMIX / trmix; wseq = wf[famTr] / wf[famTr].mean(); t0 = time.time()
    for ep in range(EP):
        net.train(); order = rng.permutation(N_TR)
        for g in range(N_TR // 32):
            k = np.sort(order[g * 32:(g + 1) * 32]); x = torch.from_numpy(getx(XA, ids[k], feats)).to(dev)
            y = torch.from_numpy(YTr[k]).to(dev).clamp(-2, 2); ws = torch.from_numpy((wseq[k] if reweight else np.ones(32)).astype(np.float32)).to(dev); h = None
            for t in range(0, L, 1000):
                o, h = net(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]
                m = (torch.arange(t, t + 1000, device=dev) >= WARM)[None].expand(len(k), -1); w3 = ws[:, None, None] * yy.abs()
                l = (w3 * (o - yy) ** 2)[m[..., None].expand_as(yy)].mean()
                if loss == 'pearson2+wmse': l = l + 1 - 0.5 * sum(wpear(o[..., j][m], yy[..., j][m], w3[..., j][m]) for j in (0, 1))
                opt.zero_grad(set_to_none=True); l.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step()
    P3 = pred(net, Xv, np.arange(300), feats); Ma = np.broadcast_to(post, (300, L)); f3 = famv == 3
    Ssc, Sall, S3 = sums(P3, Yv, Mv & Ma), sums(P3, Yv, Ma), sums(P3[f3], Yv[f3], (Mv & Ma)[f3])
    for s in range(0, len(XR), 300):
        i = np.arange(s, min(s + 300, len(XR))); Sall += sums(pred(net, XR, i, feats), YR[i], np.broadcast_to(post, (len(i), L)))
    r = {'is_scored (300)': corrT(Ssc), 'F3 is_scored': corrT(S3), 'все строки (1873)': corrT(Sall), 'минут': (time.time() - t0) / 60}
    RES2[name] = r; json.dump(RES2, open(RJ, 'w'), ensure_ascii=False, indent=1)
    print(f'{name:44s} is_scored {r["is_scored (300)"]:.4f} | F3 {r["F3 is_scored"]:.4f} | все строки (1873) {r["все строки (1873)"]:.4f} | {r["минут"]:.1f} мин', flush=True)
print('окружение готово | уже посчитано в схеме train → valid:', len(RES2), 'вариантов')

Mounted at /content/drive
GPU: Tesla T4
качаю стартпак (10–15 минут)...

скачано за 630 с
valid300 0 1 с
  validrest 0/1573 0 с
  trainall 0/10607 0 с
  trainall 2000/10607 126 с
  trainall 4000/10607 254 с
  trainall 6000/10607 384 с
  trainall 8000/10607 526 с
  trainall 10000/10607 657 с
overlay         113G   56G   58G  50% /
окружение готово | уже посчитано в схеме train → valid: 0 вариантов


In [2]:
SEED, EPF, VALID_W = 0, 5, 1.0                                  # VALID_W > 1 — давать последовательностям valid больший вес
class Cat3:                                                      # train + 1573 valid + 300 valid как один набор
    def __init__(s, parts): s.parts = parts; s.off = np.cumsum([0] + [len(p) for p in parts])
    def __len__(s): return int(s.off[-1])
    def __getitem__(s, idx):
        idx = np.sort(np.atleast_1d(idx)); out = []
        for k, p in enumerate(s.parts):
            m = (idx >= s.off[k]) & (idx < s.off[k + 1])
            if m.any(): out.append(np.asarray(p[idx[m] - s.off[k]], np.float32))
        return np.concatenate(out)
XALL = Cat3([XA, XR, Xv]); YALL = np.concatenate([np.asarray(YA), YR, Yv]); FALL = np.concatenate([famT, famR, famv])
N = len(XALL); isval = np.r_[np.zeros(len(XA)), np.ones(len(XR) + len(Xv))]
mix = np.bincount(FALL, minlength=4) / N; wseq = (VALMIX / mix)[FALL] * np.where(isval > 0, VALID_W, 1.0); wseq = (wseq / wseq.mean()).astype(np.float32)
print(f'обучающих: {N} (train {len(XA)} + valid {len(XR) + len(Xv)}) | сид {SEED} | вес valid ×{VALID_W}')
torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
net = Net().to(dev); opt = torch.optim.Adam(net.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPF * (N // 32) * (L // 1000), pct_start=0.05)
for ep in range(EPF):
    net.train(); t0 = time.time(); order = rng.permutation(N); run = []
    for g in range(N // 32):
        k = np.sort(order[g * 32:(g + 1) * 32]); x = torch.from_numpy(XALL[k]).to(dev)
        y = torch.from_numpy(YALL[k]).to(dev).clamp(-2, 2); ws = torch.from_numpy(wseq[k]).to(dev); h = None
        for t in range(0, L, 1000):
            o, h = net(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]; l = ws[:, None, None] * yy.abs() * (o - yy) ** 2
            if t < WARM: l = l[:, WARM - t:]
            opt.zero_grad(set_to_none=True); l.mean().backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step(); run.append(l.mean().item())
    torch.save(net.state_dict(), f'{CKPT_DIR}/final_s{SEED}_ep{ep + 1}.pt')
    print(f'эпоха {ep + 1}/{EPF} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | сохранено final_s{SEED}_ep{ep + 1}.pt', flush=True)
print('готово. Локально проверять нечем: все 300 отложенных теперь в обучении — проверка будет публичным сабмитом.')

обучающих: 12480 (train 10607 + valid 1873) | сид 0 | вес valid ×1.0
эпоха 1/5 | 612 с | loss 0.9122 | сохранено final_s0_ep1.pt
эпоха 2/5 | 607 с | loss 0.8819 | сохранено final_s0_ep2.pt
эпоха 3/5 | 614 с | loss 0.8711 | сохранено final_s0_ep3.pt
эпоха 4/5 | 606 с | loss 0.8581 | сохранено final_s0_ep4.pt
эпоха 5/5 | 596 с | loss nan | сохранено final_s0_ep5.pt
готово. Локально проверять нечем: все 300 отложенных теперь в обучении — проверка будет публичным сабмитом.


In [3]:
import zipfile, importlib.util
from types import SimpleNamespace
SRC, OUT = '/content/ens3_src', '/content/ens3final'
if not os.path.exists(f'{SRC}/weights.npz'):
    os.makedirs(SRC, exist_ok=True); zipfile.ZipFile(f'{CKPT_DIR}/ens3_submission.zip').extractall(SRC)

# ---------- выбираем исправную эпоху ----------
for ep in (4, 3):
    sd = torch.load(f'{CKPT_DIR}/final_s0_ep{ep}.pt', map_location='cpu')
    if all(torch.isfinite(v).all() for v in sd.values()): print(f'беру final_s0_ep{ep}.pt — веса конечные'); break
    print(f'final_s0_ep{ep}.pt испорчен (NaN в весах)')
else: raise RuntimeError('нет исправной эпохи')
net = Net(); net.load_state_dict(sd); net.eval()
def torch_out(X):
    with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32)))[0].numpy()
Pchk = np.concatenate([torch_out(Xv[i:i + 25]) for i in range(0, 300, 25)])
print('санитарная проверка на 300 valid (модель их видела, скор завышен):', round(float(np.mean([np.corrcoef(Pchk[:, WARM:, j].ravel(), Yv[:, WARM:, j].ravel())[0, 1] for j in (0, 1)])), 4),
      '| прогнозы конечны:', bool(np.isfinite(Pchk).all()))

# ---------- дописываем модель четвёртой к ens3 с равным весом ----------
def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid):
    out = np.zeros((len(X), 2), np.float32)
    for t in range(len(X)): out[t] = pm.predict(SimpleNamespace(seq_ix=sid, step_in_seq=t, state=X[t], need_prediction=True))
    return out
W0 = dict(np.load(f'{SRC}/weights.npz')); K_old = len(W0['mods']); assert K_old == 3
pm_old = load_pm(SRC, 'sol_old'); Xs = np.asarray(Xv[:20], np.float32)
old = np.stack([run_pm(pm_old, Xs[i], i) for i in range(20)]).astype(np.float64); old_nc = old - W0['const'].astype(np.float64)
Pn = np.concatenate([torch_out(Xs[i:i + 5]) for i in range(0, 20, 5)]).astype(np.float64)
st = lambda a: (a[:, WARM:].reshape(-1, 2).mean(0), a[:, WARM:].reshape(-1, 2).std(0))
mu_o, sd_o = st(old_nc); m_old = st(old)[0]; mu_p, sd_p = st(Pn)
U_ = K_old * (old_nc - mu_o) / sd_o + (Pn - mu_p) / sd_p
s = sd_o / st(U_)[1]; A = K_old * s / sd_o; w = s / sd_p; const_new = m_old - A * mu_o - w * mu_p
mats, flat, meta, mods = W0['mats'], W0['flat'], W0['meta'], W0['mods']
mo, fo, r = len(mats), len(flat), len(meta); so = int((mods[:, 2] + 2 * mods[:, 3]).max()); mc, fc, rows = [], [], []
def addm(a):
    global mo
    a = np.ascontiguousarray(a, dtype=np.float16); mc.append(a.view(np.uint16).reshape(-1)); rows.append([mo, a.shape[0], a.shape[1]]); mo += a.size
def addv(v):
    global fo
    v = np.ascontiguousarray(v, dtype=np.float32); fc.append(v); rows.append([fo, 1, len(v)]); fo += len(v)
g = lambda k: sd[k].numpy().astype(np.float32)
addm(g('gru.weight_ih_l0').T); addm(g('gru.weight_hh_l0').T); addv(g('gru.bias_ih_l0')); addv(g('gru.bias_hh_l0'))
addm(g('gru.weight_ih_l1').T); addm(g('gru.weight_hh_l1').T); addv(g('gru.bias_ih_l1')); addv(g('gru.bias_hh_l1'))
addm(g('head.weight').T); addv(g('head.bias'))
Wn = dict(W0)
Wn['mats'] = np.concatenate([mats] + mc); Wn['flat'] = np.concatenate([flat] + fc)
Wn['meta'] = np.vstack([meta, np.array(rows, np.int64)]); Wn['mods'] = np.vstack([mods, np.array([[0, r, so, 128]], np.int64)])
Wn['coef'] = np.vstack([A * W0['coef'], w[None, :]]).astype(np.float32); Wn['const'] = const_new.astype(np.float32)
Wn['selftest_y'] = (A * (W0['selftest_y'].astype(np.float64) - W0['const']) + w * torch_out(W0['selftest_x'][None])[0] + const_new).astype(np.float32)
assert all(np.isfinite(Wn[k]).all() for k in ('flat', 'coef', 'const', 'selftest_y')) and np.isfinite(Wn['mats'].view(np.float16)).all()
if os.path.exists(OUT): shutil.rmtree(OUT)
os.makedirs(OUT); np.savez(f'{OUT}/weights.npz', **Wn); shutil.copy(f'{SRC}/solution.py', f'{OUT}/solution.py')

# ---------- проверки ----------
pm_new = load_pm(OUT, 'sol_new'); print('самопроверка solution.py пройдена; моделей:', len(Wn['mods']))
worst = 0.0
for i in (20, 21):
    X = np.asarray(Xv[i], np.float32); o_old = run_pm(pm_old, X, 100 + i).astype(np.float64); o_n = run_pm(pm_new, X, 100 + i).astype(np.float64)
    worst = max(worst, np.abs(o_n - (A * (o_old - W0['const']) + w * torch_out(X[None])[0] + const_new)).max())
print('макс. расхождение с формулой:', worst, '(нужно < 1e-3)')
d1 = run_pm(pm_new, np.asarray(Xv[22], np.float32), 300); d2 = run_pm(pm_new, np.asarray(Xv[22], np.float32), 301)
print('детерминизм:', np.array_equal(d1, d2), '| конечны:', bool(np.isfinite(d1).all()), f'| обрезано на ±2: {100 * np.mean(np.abs(d1[WARM:]) > 2):.1f}%')
t0 = time.time(); run_pm(pm_old, np.asarray(Xv[23], np.float32), 400); t_old = time.time() - t0
t0 = time.time(); run_pm(pm_new, np.asarray(Xv[23], np.float32), 401); t_new = time.time() - t0
print(f'оценка на платформе ≈ {26.1 * t_new / t_old:.0f} мин')
shutil.make_archive('/content/ens3final_submission', 'zip', OUT); shutil.copy('/content/ens3final_submission.zip', f'{CKPT_DIR}/')
print('готово: /content/ens3final_submission.zip (копия на Drive)')

беру final_s0_ep4.pt — веса конечные


/tmp/ipykernel_482/1568585351.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32)))[0].numpy()


санитарная проверка на 300 valid (модель их видела, скор завышен): 0.3851 | прогнозы конечны: True
самопроверка solution.py пройдена; моделей: 4
макс. расхождение с формулой: 0.00035524162464972697 (нужно < 1e-3)
детерминизм: True | конечны: True | обрезано на ±2: 0.0%
оценка на платформе ≈ 32 мин
готово: /content/ens3final_submission.zip (копия на Drive)


In [4]:
SEED, EPF = 1, 5
rngs = np.random.default_rng(100 + SEED)
tr_ids = np.sort(rngs.choice(10607, 5000, replace=False))                       # своя пятёрка тысяч train
hold = np.sort(rngs.choice(1873, 300, replace=False))                           # свои 300 отложенных valid
held_idx = np.sort(np.random.default_rng(1).choice(1873, 300, replace=False))
rest_idx = np.array([i for i in range(1873) if i not in set(held_idx.tolist())])
pos = {int(i): ('v', k) for k, i in enumerate(held_idx)}; pos.update({int(i): ('r', k) for k, i in enumerate(rest_idx)})
vin = [i for i in range(1873) if i not in set(hold.tolist())]
src = [('a', int(i)) for i in tr_ids] + [pos[i] for i in vin]
ARR = {'a': (XA, YA), 'v': (Xv, Yv), 'r': (XR, YR)}
def getb(items):
    return (np.stack([np.asarray(ARR[a][0][k], np.float32) for a, k in items]),
            np.stack([np.asarray(ARR[a][1][k], np.float32) for a, k in items]))
famS = np.array([famT[k] if a == 'a' else (famv[k] if a == 'v' else famR[k]) for a, k in src])
N = len(src); mix = np.bincount(famS, minlength=4) / N; wseq = (VALMIX / mix)[famS]; wseq = (wseq / wseq.mean()).astype(np.float32)
hsrc = [pos[int(i)] for i in hold]
print(f'сид {SEED}: обучающих {N} (train 5000 + valid {len(vin)}, доля valid {len(vin) / N:.2f}) | отложено 300 valid')

@torch.no_grad()
def hold_score(net):
    net.eval(); S = np.zeros((2, 6))
    for s in range(0, 300, 50):
        xb, yb = getb(hsrc[s:s + 50]); x = torch.from_numpy(xb).to(dev); h = None; o_ = []
        for t in range(0, L, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
        S += sums(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy(), yb, np.broadcast_to(post, (len(xb), L)))
    net.train(); return corrT(S)

torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
net = Net().to(dev); opt = torch.optim.Adam(net.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPF * (N // 32) * (L // 1000), pct_start=0.05)
for ep in range(EPF):
    net.train(); t0 = time.time(); order = rng.permutation(N); run = []; skipped = 0
    for g in range(N // 32):
        k = order[g * 32:(g + 1) * 32]; xb, yb = getb([src[i] for i in k])
        x = torch.from_numpy(xb).to(dev); y = torch.from_numpy(yb).to(dev).clamp(-2, 2); ws = torch.from_numpy(wseq[k]).to(dev); h = None
        for t in range(0, L, 1000):
            o, h = net(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]; l = ws[:, None, None] * yy.abs() * (o - yy) ** 2
            if t < WARM: l = l[:, WARM - t:]
            l = l.mean()
            if not torch.isfinite(l):                                  # защита от NaN: пропускаем шаг
                opt.zero_grad(set_to_none=True); sched.step(); skipped += 1; h = None; continue
            opt.zero_grad(set_to_none=True); l.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step(); run.append(l.item())
    assert all(torch.isfinite(p).all() for p in net.parameters()), 'веса испорчены — бери прошлую эпоху'
    torch.save(net.state_dict(), f'{CKPT_DIR}/final5k_s{SEED}_ep{ep + 1}.pt')
    print(f'эпоха {ep + 1}/{EPF} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | пропущено шагов {skipped} | '
          f'отложенные 300 valid, все строки {hold_score(net):.4f}', flush=True)

сид 1: обучающих 6573 (train 5000 + valid 1573, доля valid 0.24) | отложено 300 valid
эпоха 1/5 | 294 с | loss 0.9099 | пропущено шагов 0 | отложенные 300 valid, все строки 0.4813
эпоха 2/5 | 288 с | loss 0.8730 | пропущено шагов 0 | отложенные 300 valid, все строки 0.4784
эпоха 3/5 | 288 с | loss 0.8604 | пропущено шагов 0 | отложенные 300 valid, все строки 0.4998
эпоха 4/5 | 287 с | loss 0.8450 | пропущено шагов 0 | отложенные 300 valid, все строки 0.5081
эпоха 5/5 | 285 с | loss 0.8319 | пропущено шагов 0 | отложенные 300 valid, все строки 0.5081


In [6]:
import shutil, zipfile, os
OUTD = f'{CKPT_DIR}/for_codex'; os.makedirs(OUTD, exist_ok=True)
shutil.copy(f'{CKPT_DIR}/lib.py', OUTD)
for z, sub in (('ens5ev_submission.zip', 'engine_ens5ev'), ('ens3_submission.zip', 'best_public_ens3')):
    zipfile.ZipFile(f'{CKPT_DIR}/{z}').extractall(f'{OUTD}/{sub}')
os.makedirs(f'{OUTD}/valid300', exist_ok=True)
for f in ('valid300_Y.npy', 'valid300_M.npy'): shutil.copy(f'/content/data/{f}', f'{OUTD}/valid300/')
PRED = {'P_gru6573_valid300.npy': 'GRU 2×128, 5000 train + 1573 valid; публично 0.64896; член лучшего ансамбля',
        'P_corr3k_valid300.npy': 'GRU 2×128, 3000 train, лосс wmse + корреляционный член',
        'Pb_valid300.npy': 'baseline организаторов, СЫРОЙ выход — перед использованием умножить на 0.3',
        'P_gruall_valid300.npy': 'GRU 2×128, весь train 10 607 + 1573 valid',
        'P_gruev_valid300.npy': 'GRU 2×128 + 8 событийных признаков (тип 2 в движке), сид 0',
        'P_gruh256_valid300.npy': 'GRU 2×256, весь train + 1573 valid'}
for f in PRED: shutil.copy(f'{CKPT_DIR}/{f}', f'{OUTD}/valid300/')
open(f'{OUTD}/README.txt', 'w').write(f'''Файлы для работы над LSTM

lib.py — функции команды: Net, Net2, train, predict, mix_score, fam_of, build и др.
  ВНИМАНИЕ: predict подаёт данные в сеть как есть; для float16-данных переводить вход во float32.

engine_ens5ev/ — актуальный numba-движок (solution.py + weights.npz): типы моделей 0 (GRU как у baseline),
  1 (GRU напарника), 2 (GRU на 120 входах = 112 + 8 событийных признаков, функция _evfeat).
  В конце solution.py — обёртка перекрёстного смешивания t0/t1; для LSTM её можно удалить.
  LSTM добавлять типом 3 (состояние h и c на два слоя = 4H, учесть при расчёте размера S).
best_public_ens3/ — лучший публичный ансамбль (0.65545): 2 модели напарника + наша GRU.

valid300/ — 300 отложенных последовательностей valid (row groups np.sort(np.random.default_rng(1).choice(1873, 300, replace=False))):
  valid300_Y.npy (300, 20000, 2) — таргеты; valid300_M.npy (300, 20000) — маска is_scored.
  Предсказания моделей, НЕ видевших эти 300 последовательностей (форма (300, 20000, 2), float32):
''' + ''.join(f'    {f} — {d}\n' for f, d in PRED.items()) + '''
  Модели напарника из лучшего ансамбля в этом списке нет: они, вероятно, видели valid, и их скор на valid300 был бы нечестным.
  Метрика: взвешенная корреляция Пирсона по строкам is_scored с шага 99, y и прогноз обрезаются до ±2, вес = |y|,
  среднее по t0 и t1 (функции sums_np / corr / ens_mix в lib.py). Различимая разница на этих 300 — от ~0.003.
  Честное среднее «baseline×0.3 + gru6573 + corr3k» на valid300 даёт ~0.661 (со смесью семейств valid).

ВАЖНО (новое): valid — отдельный срез данных, тест на него похож. Модель, видевшая valid, публично лучше на +0.0056.
  LSTM учить на train + 1573 valid (не трогая эти 300).
''')
print('готово:', OUTD); print(os.listdir(OUTD), os.listdir(f'{OUTD}/valid300'))

готово: /content/drive/MyDrive/wnn/for_codex
['lib.py', 'engine_ens5ev', 'best_public_ens3', 'valid300', 'README.txt'] ['valid300_Y.npy', 'valid300_M.npy', 'P_gru6573_valid300.npy', 'P_corr3k_valid300.npy', 'Pb_valid300.npy', 'P_gruall_valid300.npy', 'P_gruev_valid300.npy', 'P_gruh256_valid300.npy']


In [7]:
SEED, EPF, N5 = 2, 5, 5000
evrate = lambda x: (np.diff(np.asarray(x[::4, :22], np.float32), axis=0) != 0).any(1).mean()
t0 = time.time(); rateT = np.array([evrate(XA[s]) for s in range(len(XA))]); print(f'активность train посчитана: {time.time() - t0:.0f} с')
tr_ids = []
for f in range(4):                                               # самые активные внутри семейства, в пропорции семейств train
    cand = np.where(famT == f)[0]; n_f = int(round(N5 * np.mean(famT == f)))
    tr_ids.append(cand[np.argsort(-rateT[cand])[:n_f]])
tr_ids = np.sort(np.concatenate(tr_ids))
rngs = np.random.default_rng(100 + SEED); hold = np.sort(rngs.choice(1873, 300, replace=False))
held_idx = np.sort(np.random.default_rng(1).choice(1873, 300, replace=False))
rest_idx = np.array([i for i in range(1873) if i not in set(held_idx.tolist())])
pos = {int(i): ('v', k) for k, i in enumerate(held_idx)}; pos.update({int(i): ('r', k) for k, i in enumerate(rest_idx)})
vin = [i for i in range(1873) if i not in set(hold.tolist())]
src = [('a', int(i)) for i in tr_ids] + [pos[i] for i in vin]
ARR = {'a': (XA, YA), 'v': (Xv, Yv), 'r': (XR, YR)}
def getb(items):
    return (np.stack([np.asarray(ARR[a][0][k], np.float32) for a, k in items]),
            np.stack([np.asarray(ARR[a][1][k], np.float32) for a, k in items]))
famS = np.array([famT[k] if a == 'a' else (famv[k] if a == 'v' else famR[k]) for a, k in src])
N = len(src); mix = np.bincount(famS, minlength=4) / N; wseq = (VALMIX / mix)[famS]; wseq = (wseq / wseq.mean()).astype(np.float32)
hsrc = [pos[int(i)] for i in hold]
print(f'сид {SEED}: train {len(tr_ids)} (активность {rateT[tr_ids].mean():.3f} против {rateT.mean():.3f} в среднем) + valid {len(vin)} | отложено 300 valid')

@torch.no_grad()
def hold_score(net):
    net.eval(); S = np.zeros((2, 6))
    for s in range(0, 300, 50):
        xb, yb = getb(hsrc[s:s + 50]); x = torch.from_numpy(xb).to(dev); h = None; o_ = []
        for t in range(0, L, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
        S += sums(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy(), yb, np.broadcast_to(post, (len(xb), L)))
    net.train(); return corrT(S)

torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
net = Net().to(dev); opt = torch.optim.Adam(net.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPF * (N // 32) * (L // 1000), pct_start=0.05)
for ep in range(EPF):
    net.train(); t0 = time.time(); order = rng.permutation(N); run = []; skipped = 0
    for g in range(N // 32):
        k = order[g * 32:(g + 1) * 32]; xb, yb = getb([src[i] for i in k])
        x = torch.from_numpy(xb).to(dev); y = torch.from_numpy(yb).to(dev).clamp(-2, 2); ws = torch.from_numpy(wseq[k]).to(dev); h = None
        for t in range(0, L, 1000):
            o, h = net(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]; l = ws[:, None, None] * yy.abs() * (o - yy) ** 2
            if t < WARM: l = l[:, WARM - t:]
            l = l.mean()
            if not torch.isfinite(l): opt.zero_grad(set_to_none=True); sched.step(); skipped += 1; h = None; continue
            opt.zero_grad(set_to_none=True); l.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step(); run.append(l.item())
    assert all(torch.isfinite(p).all() for p in net.parameters()), 'веса испорчены — бери прошлую эпоху'
    torch.save(net.state_dict(), f'{CKPT_DIR}/final5kact_s{SEED}_ep{ep + 1}.pt')
    print(f'эпоха {ep + 1}/{EPF} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | пропущено {skipped} | '
          f'отложенные 300 valid, все строки {hold_score(net):.4f}', flush=True)

активность train посчитана: 190 с
сид 2: train 5000 (активность 0.380 против 0.284 в среднем) + valid 1573 | отложено 300 valid
эпоха 1/5 | 293 с | loss 0.8396 | пропущено 0 | отложенные 300 valid, все строки 0.4849
эпоха 2/5 | 289 с | loss 0.8014 | пропущено 0 | отложенные 300 valid, все строки 0.4929
эпоха 3/5 | 288 с | loss 0.7891 | пропущено 0 | отложенные 300 valid, все строки 0.4993
эпоха 4/5 | 289 с | loss 0.7743 | пропущено 0 | отложенные 300 valid, все строки 0.5035
эпоха 5/5 | 285 с | loss 0.7621 | пропущено 0 | отложенные 300 valid, все строки 0.5046


In [8]:
import zipfile, importlib.util
from types import SimpleNamespace
BASE_ZIP, OUT_NAME = 'ens3final_submission.zip', 'ens5act'       # от какого архива строим и как назовём новый
NEW = ['final5kact_s2_ep5.pt']                                   # какие модели добавить (тип 0, GRU 2×128)
BASE_MIN = 34.2                                                  # реальное время базового архива на платформе, мин
SRC, OUT = f'/content/{BASE_ZIP[:-4]}_src', f'/content/{OUT_NAME}'
if os.path.exists(SRC): shutil.rmtree(SRC)
os.makedirs(SRC); zipfile.ZipFile(f'{CKPT_DIR}/{BASE_ZIP}').extractall(SRC)

def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid):
    out = np.zeros((len(X), 2), np.float32)
    for t in range(len(X)): out[t] = pm.predict(SimpleNamespace(seq_ix=sid, step_in_seq=t, state=X[t], need_prediction=True))
    return out
st = lambda a: (a[:, WARM:].reshape(-1, 2).mean(0), a[:, WARM:].reshape(-1, 2).std(0))
Xs = np.asarray(Xv[:20], np.float32)

W = dict(np.load(f'{SRC}/weights.npz'))
for ck in NEW:
    sd = torch.load(f'{CKPT_DIR}/{ck}', map_location='cpu'); assert all(torch.isfinite(v).all() for v in sd.values()), f'{ck}: NaN в весах'
    net = Net(); net.load_state_dict(sd); net.eval()
    tout = lambda X: net(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32)))[0].detach().numpy()
    np.savez(f'{OUT}_tmp.npz', **W); os.makedirs(f'{OUT}_tmp', exist_ok=True)
    shutil.copy(f'{SRC}/solution.py', f'{OUT}_tmp/solution.py'); shutil.move(f'{OUT}_tmp.npz', f'{OUT}_tmp/weights.npz')
    pm_cur = load_pm(f'{OUT}_tmp', f'sol_cur_{ck}'); K = len(W['mods'])
    old = np.stack([run_pm(pm_cur, Xs[i], i) for i in range(20)]).astype(np.float64); old_nc = old - W['const'].astype(np.float64)
    with torch.no_grad(): Pn = np.concatenate([tout(Xs[i:i + 5]) for i in range(0, 20, 5)]).astype(np.float64)
    mu_o, sd_o = st(old_nc); m_old = st(old)[0]; mu_p, sd_p = st(Pn)
    U_ = K * (old_nc - mu_o) / sd_o + (Pn - mu_p) / sd_p; s = sd_o / st(U_)[1]; A = K * s / sd_o; w = s / sd_p
    const_new = m_old - A * mu_o - w * mu_p
    mats, flat, meta, mods = W['mats'], W['flat'], W['meta'], W['mods']
    mo, fo, r = len(mats), len(flat), len(meta); so = int((mods[:, 2] + 2 * mods[:, 3]).max()); mc, fc, rows = [], [], []
    def addm(a):
        global mo
        a = np.ascontiguousarray(a, dtype=np.float16); mc.append(a.view(np.uint16).reshape(-1)); rows.append([mo, a.shape[0], a.shape[1]]); mo += a.size
    def addv(v):
        global fo
        v = np.ascontiguousarray(v, dtype=np.float32); fc.append(v); rows.append([fo, 1, len(v)]); fo += len(v)
    g = lambda k: sd[k].numpy().astype(np.float32)
    addm(g('gru.weight_ih_l0').T); addm(g('gru.weight_hh_l0').T); addv(g('gru.bias_ih_l0')); addv(g('gru.bias_hh_l0'))
    addm(g('gru.weight_ih_l1').T); addm(g('gru.weight_hh_l1').T); addv(g('gru.bias_ih_l1')); addv(g('gru.bias_hh_l1'))
    addm(g('head.weight').T); addv(g('head.bias'))
    with torch.no_grad(): pt = tout(W['selftest_x'][None])[0].astype(np.float64)
    Wn = dict(W)
    Wn['mats'] = np.concatenate([mats] + mc); Wn['flat'] = np.concatenate([flat] + fc)
    Wn['meta'] = np.vstack([meta, np.array(rows, np.int64)]); Wn['mods'] = np.vstack([mods, np.array([[0, r, so, 128]], np.int64)])
    Wn['coef'] = np.vstack([A * W['coef'], w[None, :]]).astype(np.float32); Wn['const'] = const_new.astype(np.float32)
    Wn['selftest_y'] = (A * (W['selftest_y'].astype(np.float64) - W['const']) + w * pt + const_new).astype(np.float32)
    W = Wn; print(f'добавлена {ck}: моделей теперь {len(W["mods"])}')
    shutil.rmtree(f'{OUT}_tmp')

assert all(np.isfinite(W[k]).all() for k in ('flat', 'coef', 'const', 'selftest_y'))
if os.path.exists(OUT): shutil.rmtree(OUT)
os.makedirs(OUT); np.savez(f'{OUT}/weights.npz', **W); shutil.copy(f'{SRC}/solution.py', f'{OUT}/solution.py')

# ---------- проверки ----------
pm_base, pm_new = load_pm(SRC, 'sol_base'), load_pm(OUT, 'sol_new'); print('самопроверка solution.py пройдена; моделей:', len(W['mods']))
X = np.asarray(Xv[22], np.float32)
d1, d2 = run_pm(pm_new, X, 300), run_pm(pm_new, X, 301)
print('детерминизм:', np.array_equal(d1, d2), '| конечны:', bool(np.isfinite(d1).all()), f'| обрезано на ±2: {100 * np.mean(np.abs(d1[WARM:]) > 2):.1f}%')
ob = run_pm(pm_base, X, 302)
print('корреляция с базовым ансамблем по строкам:', round(float(np.corrcoef(ob[WARM:].ravel(), d1[WARM:].ravel())[0, 1]), 4), '(ожидаю 0.98–0.995)')
t0 = time.time(); run_pm(pm_base, np.asarray(Xv[23], np.float32), 400); t_b = time.time() - t0
t0 = time.time(); run_pm(pm_new, np.asarray(Xv[23], np.float32), 401); t_n = time.time() - t0
print(f'оценка на платформе ≈ {BASE_MIN * t_n / t_b:.0f} мин (лимит 60)')
shutil.make_archive(f'/content/{OUT_NAME}_submission', 'zip', OUT); shutil.copy(f'/content/{OUT_NAME}_submission.zip', f'{CKPT_DIR}/')
print(f'готово: /content/{OUT_NAME}_submission.zip (копия на Drive)')

добавлена final5kact_s2_ep5.pt: моделей теперь 5
самопроверка solution.py пройдена; моделей: 5
детерминизм: True | конечны: True | обрезано на ±2: 0.0%
корреляция с базовым ансамблем по строкам: 0.9969 (ожидаю 0.98–0.995)
оценка на платформе ≈ 42 мин (лимит 60)
готово: /content/ens5act_submission.zip (копия на Drive)


In [9]:
# ===== Сборка ens5act = ens3final (0.65614) + final5kact_s2 =====
import zipfile, importlib.util
from types import SimpleNamespace
BASE_ZIP, OUT_NAME = 'ens3final_submission.zip', 'ens5act'       # основа и имя нового архива
NEW = ['final5kact_s2_ep5.pt']                                   # какие модели добавить (GRU 2×128, тип 0)
BASE_MIN = 34.2                                                  # реальное время основы на платформе, мин
SRC, OUT = '/content/base_src', f'/content/{OUT_NAME}'
for d in (SRC, OUT, f'{OUT}_tmp'):
    if os.path.exists(d): shutil.rmtree(d)
os.makedirs(SRC); zipfile.ZipFile(f'{CKPT_DIR}/{BASE_ZIP}').extractall(SRC)

def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid):
    out = np.zeros((len(X), 2), np.float32)
    for t in range(len(X)): out[t] = pm.predict(SimpleNamespace(seq_ix=sid, step_in_seq=t, state=X[t], need_prediction=True))
    return out
st = lambda a: (a[:, WARM:].reshape(-1, 2).mean(0), a[:, WARM:].reshape(-1, 2).std(0))
Xs = np.asarray(Xv[:20], np.float32)

W = dict(np.load(f'{SRC}/weights.npz')); print('моделей в основе:', len(W['mods']))
for n_i, ck in enumerate(NEW):
    sd = torch.load(f'{CKPT_DIR}/{ck}', map_location='cpu'); assert all(torch.isfinite(v).all() for v in sd.values()), f'{ck}: NaN в весах'
    net = Net(); net.load_state_dict(sd); net.eval()
    def tout(X):
        with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32)))[0].numpy().astype(np.float64)
    # текущий ансамбль — из W (основа плюс уже добавленные модели)
    os.makedirs(f'{OUT}_tmp'); np.savez(f'{OUT}_tmp/weights.npz', **W); shutil.copy(f'{SRC}/solution.py', f'{OUT}_tmp/solution.py')
    pm_cur = load_pm(f'{OUT}_tmp', f'sol_cur_{n_i}'); K = len(W['mods'])
    old = np.stack([run_pm(pm_cur, Xs[i], i) for i in range(20)]).astype(np.float64); old_nc = old - W['const'].astype(np.float64)
    Pn = np.concatenate([tout(Xs[i:i + 5]) for i in range(0, 20, 5)])
    mu_o, sd_o = st(old_nc); m_old = st(old)[0]; mu_p, sd_p = st(Pn)
    U_ = K * (old_nc - mu_o) / sd_o + (Pn - mu_p) / sd_p; s = sd_o / st(U_)[1]; A = K * s / sd_o; w = s / sd_p
    const_new = m_old - A * mu_o - w * mu_p                     # каждая модель получает вес 1/(K+1)
    mats, flat, meta, mods = W['mats'], W['flat'], W['meta'], W['mods']
    mo, fo, r = len(mats), len(flat), len(meta); so = int((mods[:, 2] + 2 * mods[:, 3]).max()); mc, fc, rows = [], [], []
    def addm(a):
        global mo
        a = np.ascontiguousarray(a, dtype=np.float16); mc.append(a.view(np.uint16).reshape(-1)); rows.append([mo, a.shape[0], a.shape[1]]); mo += a.size
    def addv(v):
        global fo
        v = np.ascontiguousarray(v, dtype=np.float32); fc.append(v); rows.append([fo, 1, len(v)]); fo += len(v)
    g = lambda k: sd[k].numpy().astype(np.float32)
    addm(g('gru.weight_ih_l0').T); addm(g('gru.weight_hh_l0').T); addv(g('gru.bias_ih_l0')); addv(g('gru.bias_hh_l0'))
    addm(g('gru.weight_ih_l1').T); addm(g('gru.weight_hh_l1').T); addv(g('gru.bias_ih_l1')); addv(g('gru.bias_hh_l1'))
    addm(g('head.weight').T); addv(g('head.bias'))
    pt = tout(W['selftest_x'][None])[0]
    Wn = dict(W)
    Wn['mats'] = np.concatenate([mats] + mc); Wn['flat'] = np.concatenate([flat] + fc)
    Wn['meta'] = np.vstack([meta, np.array(rows, np.int64)]); Wn['mods'] = np.vstack([mods, np.array([[0, r, so, 128]], np.int64)])
    Wn['coef'] = np.vstack([A * W['coef'], w[None, :]]).astype(np.float32); Wn['const'] = const_new.astype(np.float32)
    Wn['selftest_y'] = (A * (W['selftest_y'].astype(np.float64) - W['const']) + w * pt + const_new).astype(np.float32)
    W = Wn; shutil.rmtree(f'{OUT}_tmp'); print(f'добавлена {ck}: моделей теперь {len(W["mods"])}')

assert all(np.isfinite(W[k]).all() for k in ('flat', 'coef', 'const', 'selftest_y')), 'NaN в весах ансамбля'
os.makedirs(OUT); np.savez(f'{OUT}/weights.npz', **W); shutil.copy(f'{SRC}/solution.py', f'{OUT}/solution.py')

# ---------- проверки ----------
pm_base, pm_new = load_pm(SRC, 'sol_base_chk'), load_pm(OUT, 'sol_new_chk'); print('самопроверка solution.py пройдена; моделей:', len(W['mods']))
X = np.asarray(Xv[22], np.float32)
d1, d2 = run_pm(pm_new, X, 300), run_pm(pm_new, X, 301)
print('детерминизм:', np.array_equal(d1, d2), '| конечны:', bool(np.isfinite(d1).all()), f'| обрезано на ±2: {100 * np.mean(np.abs(d1[WARM:]) > 2):.1f}%')
ob = run_pm(pm_base, X, 302)
print('корреляция с основой по строкам:', round(float(np.corrcoef(ob[WARM:].ravel(), d1[WARM:].ravel())[0, 1]), 4), '(ожидаю 0.98–0.995)')
t0 = time.time(); run_pm(pm_base, np.asarray(Xv[23], np.float32), 400); t_b = time.time() - t0
t0 = time.time(); run_pm(pm_new, np.asarray(Xv[23], np.float32), 401); t_n = time.time() - t0
print(f'оценка на платформе ≈ {BASE_MIN * t_n / t_b:.0f} мин (лимит 60)')
shutil.make_archive(f'/content/{OUT_NAME}_submission', 'zip', OUT); shutil.copy(f'/content/{OUT_NAME}_submission.zip', f'{CKPT_DIR}/')
print(f'готово: /content/{OUT_NAME}_submission.zip (копия на Drive)')

моделей в основе: 4
добавлена final5kact_s2_ep5.pt: моделей теперь 5
самопроверка solution.py пройдена; моделей: 5
детерминизм: True | конечны: True | обрезано на ±2: 0.0%
корреляция с основой по строкам: 0.9969 (ожидаю 0.98–0.995)
оценка на платформе ≈ 41 мин (лимит 60)
готово: /content/ens5act_submission.zip (копия на Drive)


In [10]:
SEED, EPF, N5 = 3, 5, 5000                                       # на третьем аккаунте поставь SEED = 4
evrate = lambda x: (np.diff(np.asarray(x[::4, :22], np.float32), axis=0) != 0).any(1).mean()
if 'rateT' not in globals() or len(rateT) != len(XA):
    t0 = time.time(); rateT = np.array([evrate(XA[s]) for s in range(len(XA))]); print(f'активность train посчитана: {time.time() - t0:.0f} с')
tr_ids = np.sort(np.concatenate([np.where(famT == f)[0][np.argsort(-rateT[famT == f])][:int(round(N5 * np.mean(famT == f)))] for f in range(4)]))
rngs = np.random.default_rng(100 + SEED); hold = np.sort(rngs.choice(1873, 300, replace=False))        # свои 300 отложенных valid
held_idx = np.sort(np.random.default_rng(1).choice(1873, 300, replace=False))
rest_idx = np.array([i for i in range(1873) if i not in set(held_idx.tolist())])
pos = {int(i): ('v', k) for k, i in enumerate(held_idx)}; pos.update({int(i): ('r', k) for k, i in enumerate(rest_idx)})
vin = [i for i in range(1873) if i not in set(hold.tolist())]
src = [('a', int(i)) for i in tr_ids] + [pos[i] for i in vin]
ARR = {'a': (XA, YA), 'v': (Xv, Yv), 'r': (XR, YR)}
def getb(items):
    return (np.stack([np.asarray(ARR[a][0][k], np.float32) for a, k in items]),
            np.stack([np.asarray(ARR[a][1][k], np.float32) for a, k in items]))
famS = np.array([famT[k] if a == 'a' else (famv[k] if a == 'v' else famR[k]) for a, k in src])
N = len(src); mix = np.bincount(famS, minlength=4) / N; wseq = (VALMIX / mix)[famS]; wseq = (wseq / wseq.mean()).astype(np.float32)
hsrc = [pos[int(i)] for i in hold]
print(f'сид {SEED}: train {len(tr_ids)} (активность {rateT[tr_ids].mean():.3f} против {rateT.mean():.3f}) + valid {len(vin)} | отложено 300 valid')

@torch.no_grad()
def hold_score(net):
    net.eval(); S = np.zeros((2, 6))
    for s in range(0, 300, 50):
        xb, yb = getb(hsrc[s:s + 50]); x = torch.from_numpy(xb).to(dev); h = None; o_ = []
        for t in range(0, L, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
        S += sums(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy(), yb, np.broadcast_to(post, (len(xb), L)))
    net.train(); return corrT(S)

torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
net = Net().to(dev); opt = torch.optim.Adam(net.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPF * (N // 32) * (L // 1000), pct_start=0.05)
for ep in range(EPF):
    net.train(); t0 = time.time(); order = rng.permutation(N); run = []; skipped = 0
    for g in range(N // 32):
        k = order[g * 32:(g + 1) * 32]; xb, yb = getb([src[i] for i in k])
        x = torch.from_numpy(xb).to(dev); y = torch.from_numpy(yb).to(dev).clamp(-2, 2); ws = torch.from_numpy(wseq[k]).to(dev); h = None
        for t in range(0, L, 1000):
            o, h = net(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]; l = ws[:, None, None] * yy.abs() * (o - yy) ** 2
            if t < WARM: l = l[:, WARM - t:]
            l = l.mean()
            if not torch.isfinite(l): opt.zero_grad(set_to_none=True); sched.step(); skipped += 1; h = None; continue
            opt.zero_grad(set_to_none=True); l.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step(); run.append(l.item())
    assert all(torch.isfinite(p).all() for p in net.parameters()), 'веса испорчены — бери прошлую эпоху'
    torch.save(net.state_dict(), f'{CKPT_DIR}/final5kact_s{SEED}_ep{ep + 1}.pt')
    print(f'эпоха {ep + 1}/{EPF} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | пропущено {skipped} | '
          f'отложенные 300 valid, все строки {hold_score(net):.4f}', flush=True)

сид 3: train 5000 (активность 0.380 против 0.284) + valid 1573 | отложено 300 valid
эпоха 1/5 | 293 с | loss 0.8363 | пропущено 0 | отложенные 300 valid, все строки 0.4855
эпоха 2/5 | 294 с | loss 0.7985 | пропущено 0 | отложенные 300 valid, все строки 0.4900
эпоха 3/5 | 292 с | loss 0.7864 | пропущено 0 | отложенные 300 valid, все строки 0.4960
эпоха 4/5 | 290 с | loss 0.7720 | пропущено 0 | отложенные 300 valid, все строки 0.5049
эпоха 5/5 | 287 с | loss 0.7599 | пропущено 0 | отложенные 300 valid, все строки 0.5041


In [2]:
SEED, EPF, N5 = 0, 5, 5000
evrate = lambda x: (np.diff(np.asarray(x[::4, :22], np.float32), axis=0) != 0).any(1).mean()
if 'rateT' not in globals() or len(rateT) != len(XA):
    t0 = time.time(); rateT = np.array([evrate(XA[s]) for s in range(len(XA))]); print(f'активность train: {time.time() - t0:.0f} с')
tr_ids = np.sort(np.concatenate([np.where(famT == f)[0][np.argsort(-rateT[famT == f])][:int(round(N5 * np.mean(famT == f)))] for f in range(4)]))
rngs = np.random.default_rng(200 + SEED); hold = np.sort(rngs.choice(1873, 300, replace=False))
held_idx = np.sort(np.random.default_rng(1).choice(1873, 300, replace=False))
rest_idx = np.array([i for i in range(1873) if i not in set(held_idx.tolist())])
pos = {int(i): ('v', k) for k, i in enumerate(held_idx)}; pos.update({int(i): ('r', k) for k, i in enumerate(rest_idx)})
vin = [i for i in range(1873) if i not in set(hold.tolist())]
src = [('a', int(i)) for i in tr_ids] + [pos[i] for i in vin]
ARR = {'a': (XA, YA), 'v': (Xv, Yv), 'r': (XR, YR)}
def getb(items):
    return (np.stack([np.asarray(ARR[a][0][k], np.float32) for a, k in items]), np.stack([np.asarray(ARR[a][1][k], np.float32) for a, k in items]))
famS = np.array([famT[k] if a == 'a' else (famv[k] if a == 'v' else famR[k]) for a, k in src])
N = len(src); mix = np.bincount(famS, minlength=4) / N; wseq = (VALMIX / mix)[famS]; wseq = (wseq / wseq.mean()).astype(np.float32)
hsrc = [pos[int(i)] for i in hold]

class NetL(nn.Module):                                           # LSTM 2×128 + линейная голова
    def __init__(s, H=128): super().__init__(); s.rnn = nn.LSTM(112, H, 2, batch_first=True); s.head = nn.Linear(H, 2)
    def forward(s, x, h=None): y, h = s.rnn(x, h); return s.head(y), h
@torch.no_grad()
def hold_score(net):
    net.eval(); S = np.zeros((2, 6))
    for s in range(0, 300, 50):
        xb, yb = getb(hsrc[s:s + 50]); x = torch.from_numpy(xb).to(dev); h = None; o_ = []
        for t in range(0, L, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
        S += sums(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy(), yb, np.broadcast_to(post, (len(xb), L)))
    net.train(); return corrT(S)
print(f'LSTM, сид {SEED}: train {len(tr_ids)} самых активных + valid {len(vin)} | отложено 300 valid')
torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
net = NetL().to(dev); opt = torch.optim.Adam(net.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPF * (N // 32) * (L // 1000), pct_start=0.05)
for ep in range(EPF):
    net.train(); t0 = time.time(); order = rng.permutation(N); run = []; skipped = 0
    for g in range(N // 32):
        k = order[g * 32:(g + 1) * 32]; xb, yb = getb([src[i] for i in k])
        x = torch.from_numpy(xb).to(dev); y = torch.from_numpy(yb).to(dev).clamp(-2, 2); ws = torch.from_numpy(wseq[k]).to(dev); h = None
        for t in range(0, L, 1000):
            o, h = net(x[:, t:t + 1000], h); h = tuple(v.detach() for v in h); yy = y[:, t:t + 1000]
            l = ws[:, None, None] * yy.abs() * (o - yy) ** 2
            if t < WARM: l = l[:, WARM - t:]
            l = l.mean()
            if not torch.isfinite(l): opt.zero_grad(set_to_none=True); sched.step(); skipped += 1; h = None; continue
            opt.zero_grad(set_to_none=True); l.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step(); run.append(l.item())
    assert all(torch.isfinite(p).all() for p in net.parameters()), 'веса испорчены — бери прошлую эпоху'
    torch.save(net.state_dict(), f'{CKPT_DIR}/final5kact_lstm_s{SEED}_ep{ep + 1}.pt')
    print(f'эпоха {ep + 1}/{EPF} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | пропущено {skipped} | '
          f'отложенные 300 valid, все строки {hold_score(net):.4f}', flush=True)

активность train: 191 с
LSTM, сид 0: train 5000 самых активных + valid 1573 | отложено 300 valid
эпоха 1/5 | 329 с | loss 0.8494 | пропущено 0 | отложенные 300 valid, все строки 0.4897
эпоха 2/5 | 336 с | loss 0.8024 | пропущено 0 | отложенные 300 valid, все строки 0.4980
эпоха 3/5 | 337 с | loss 0.7878 | пропущено 0 | отложенные 300 valid, все строки 0.5056
эпоха 4/5 | 339 с | loss 0.7761 | пропущено 0 | отложенные 300 valid, все строки 0.5095
эпоха 5/5 | 343 с | loss 0.7670 | пропущено 0 | отложенные 300 valid, все строки 0.5106


In [5]:
import zipfile, importlib.util
from types import SimpleNamespace
# ---------- доработка движка: тип 3 = LSTM (веса во float32) ----------
LSTM_FUNCS = '''@njit(**_KW)
def _matf(flat, meta, r):
    """A float32 matrix stored in flat (used by type 3: LSTM weights are kept in float32)."""
    o, n, m = meta[r, 0], meta[r, 1], meta[r, 2]
    return flat[o:o + n * m].reshape((n, m))


@njit(**_KW)
def _lstm(x, h, c, WiT, WhT, bi, bh, gi, gh):
    """PyTorch LSTM cell (gate order i, f, g, o); updates h and c in place."""
    H = h.shape[0]
    _gemv(x, WiT, bi, gi)
    _gemv(h, WhT, bh, gh)
    for j in range(H):
        ig = _sigmoid(gi[j] + gh[j])
        fg = _sigmoid(gi[H + j] + gh[H + j])
        gg = _tanh(gi[2 * H + j] + gh[2 * H + j])
        og = _sigmoid(gi[3 * H + j] + gh[3 * H + j])
        c[j] = fg * c[j] + ig * gg
        h[j] = og * _tanh(c[j])


'''
LSTM_BRANCH = '''        elif typ == 3:                                                     # LSTM(112->H) -> LSTM(H->H) -> Linear(H->2), float32
            c0 = S[so + 2 * H:so + 3 * H]
            c1 = S[so + 3 * H:so + 4 * H]
            g4i = gi[:4 * H]
            g4h = gh[:4 * H]
            _lstm(x, h0, c0, _matf(flat, meta, r), _matf(flat, meta, r + 1), _vec(flat, meta, r + 2), _vec(flat, meta, r + 3), g4i, g4h)
            _lstm(h0, h1, c1, _matf(flat, meta, r + 4), _matf(flat, meta, r + 5), _vec(flat, meta, r + 6), _vec(flat, meta, r + 7), g4i, g4h)
            _gemv(h1, _matf(flat, meta, r + 8), _vec(flat, meta, r + 9), o2)
'''
def patch_lstm(src):
    if '_lstm(' in src: return src
    P = [('@njit(**_KW)\ndef _step(', LSTM_FUNCS + '@njit(**_KW)\ndef _step('),
         ('        else:\n            nx = x.shape[0]\n', LSTM_BRANCH + '        else:\n            nx = x.shape[0]\n'),
         ('        self.S = np.zeros(int((self.mods[:, 2] + 2 * self.mods[:, 3]).max()), np.float32)\n',
          '        self.S = np.zeros(int((self.mods[:, 2] + np.where(self.mods[:, 0] == 3, 4, 2) * self.mods[:, 3]).max()), np.float32)\n'),
         ('        self.gi, self.gh = np.zeros(3 * Hm, np.float32), np.zeros(3 * Hm, np.float32)\n',
          '        self.gi, self.gh = np.zeros(4 * Hm, np.float32), np.zeros(4 * Hm, np.float32)\n')]
    for a, b in P:
        assert src.count(a) == 1, f'не нашёл место для правки: {a[:60]!r}'
        src = src.replace(a, b)
    return src
state_end = lambda mods: int(max(int(so) + (4 if int(t) == 3 else 2) * int(H) for t, _, so, H in mods))

# ---------- общие функции ----------
def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid):
    out = np.zeros((len(X), 2), np.float32)
    for t in range(len(X)): out[t] = pm.predict(SimpleNamespace(seq_ix=sid, step_in_seq=t, state=X[t], need_prediction=True))
    return out
st = lambda a: (a[:, WARM:].reshape(-1, 2).mean(0), a[:, WARM:].reshape(-1, 2).std(0))
Xs = np.asarray(Xv[:20], np.float32); _cnt = [0]
def fresh_name(): _cnt[0] += 1; return f'sol_l_{_cnt[0]}'
B = '/content/ens3_base'
if os.path.exists(B): shutil.rmtree(B)
os.makedirs(B); zipfile.ZipFile(f'{CKPT_DIR}/ens3_submission.zip').extractall(B)
SOLSRC = patch_lstm(open(f'{B}/solution.py').read())
def save_dir(W, d):
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d); np.savez(f'{d}/weights.npz', **W); open(f'{d}/solution.py', 'w').write(SOLSRC)
NR = {0: 10, 2: 10, 1: 16, 3: 10}; MAT = {0: {0, 1, 4, 5, 8}, 2: {0, 1, 4, 5, 8}, 1: {2, 4, 5, 8, 9, 12, 14}, 3: set()}
def subset(W, keep):
    ml, fl, me, mo_ = [], [], [], []; mo = fo = so = 0
    for k in keep:
        typ, r, _, H = (int(v) for v in W['mods'][k]); nr = len(me)
        for j in range(NR[typ]):
            o, n, m = (int(v) for v in W['meta'][r + j])
            if j in MAT[typ]: ml.append(W['mats'][o:o + n * m]); me.append([mo, n, m]); mo += n * m
            else: fl.append(W['flat'][o:o + n * m]); me.append([fo, n, m]); fo += n * m
        mo_.append([typ, nr, so, H]); so += (4 if typ == 3 else 2) * H
    Wn = dict(W); Wn.update(mats=np.concatenate(ml) if ml else W['mats'][:0], flat=np.concatenate(fl), meta=np.array(me, np.int64),
                            mods=np.array(mo_, np.int64), coef=W['coef'][keep].astype(np.float32))
    return Wn
def fix_selftest(W):
    Wt = dict(W); Wt['selftest_x'] = W['selftest_x'][:0]; Wt['selftest_y'] = W['selftest_y'][:0]
    save_dir(Wt, '/content/_st'); pm = load_pm('/content/_st', fresh_name())
    W = dict(W); W['selftest_y'] = run_pm(pm, np.asarray(W['selftest_x'], np.float32), 12345); return W

class NetL(nn.Module):
    def __init__(s, H=128): super().__init__(); s.rnn = nn.LSTM(112, H, 2, batch_first=True); s.head = nn.Linear(H, 2)
    def forward(s, x, h=None): y, h = s.rnn(x, h); return s.head(y), h
def add_model(W, ck, kind):                                      # kind: 'gru' (тип 0, float16) или 'lstm' (тип 3, float32); равные веса
    sd = torch.load(f'{CKPT_DIR}/{ck}', map_location='cpu'); assert all(torch.isfinite(v).all() for v in sd.values()), ck
    net = (NetL() if kind == 'lstm' else Net()); net.load_state_dict(sd); net.eval()
    def tout(X_):
        with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X_, dtype=np.float32)))[0].numpy().astype(np.float64)
    save_dir(W, '/content/_cur'); pm = load_pm('/content/_cur', fresh_name()); K = len(W['mods'])
    old = np.stack([run_pm(pm, Xs[i], i) for i in range(20)]).astype(np.float64); old_nc = old - W['const'].astype(np.float64)
    Pn = np.concatenate([tout(Xs[i:i + 5]) for i in range(0, 20, 5)])
    mu_o, sd_o = st(old_nc); m_old = st(old)[0]; mu_p, sd_p = st(Pn)
    U_ = K * (old_nc - mu_o) / sd_o + (Pn - mu_p) / sd_p; s = sd_o / st(U_)[1]; A = K * s / sd_o; w = s / sd_p
    g = lambda k: np.ascontiguousarray(sd[k].numpy(), np.float32)
    mats, flat, meta = W['mats'], W['flat'], W['meta']; mo, fo, r = len(mats), len(flat), len(meta); mc, fc, rows = [], [], []
    pre = 'rnn.' if kind == 'lstm' else 'gru.'
    for a in (g(pre + 'weight_ih_l0').T, g(pre + 'weight_hh_l0').T, g(pre + 'bias_ih_l0'), g(pre + 'bias_hh_l0'),
              g(pre + 'weight_ih_l1').T, g(pre + 'weight_hh_l1').T, g(pre + 'bias_ih_l1'), g(pre + 'bias_hh_l1'), g('head.weight').T, g('head.bias')):
        if a.ndim == 2 and kind == 'gru':
            a = np.ascontiguousarray(a, np.float16); mc.append(a.view(np.uint16).reshape(-1)); rows.append([mo, a.shape[0], a.shape[1]]); mo += a.size
        else:
            a = np.ascontiguousarray(a, np.float32); fc.append(a.reshape(-1)); rows.append([fo, a.shape[0], a.shape[1]] if a.ndim == 2 else [fo, 1, a.shape[0]]); fo += a.size
    Wn = dict(W)
    if mc: Wn['mats'] = np.concatenate([mats] + mc)
    Wn['flat'] = np.concatenate([flat] + fc).astype(np.float32); Wn['meta'] = np.vstack([meta, np.array(rows, np.int64)])
    Wn['mods'] = np.vstack([W['mods'], np.array([[3 if kind == 'lstm' else 0, r, state_end(W['mods']), sd[pre + 'weight_hh_l0'].shape[1]]], np.int64)])
    Wn['coef'] = np.vstack([A * W['coef'], w[None, :]]).astype(np.float32); Wn['const'] = (m_old - A * mu_o - w * mu_p).astype(np.float32)
    # проверка: выход движка с новой моделью = A·(старый − const) + w·PyTorch + const
    Wn = fix_selftest(Wn); save_dir(Wn, '/content/_chk'); pmn = load_pm('/content/_chk', fresh_name()); X = np.asarray(Xv[24], np.float32)
    err = np.abs(run_pm(pmn, X, 7).astype(np.float64) - (A * (run_pm(pm, X, 7) - W['const']) + w * tout(X[None])[0] + Wn['const'])).max()
    print(f'  + {ck} ({kind}): моделей {len(Wn["mods"])} | расхождение движка с PyTorch на полной последовательности {err:.1e} (нужно < 1e-3)')
    assert err < 1e-3
    return Wn

# ---------- варианты ----------
VARIANTS = {'final5_lstm': ('final5_act3_submission.zip', [0, 1, 2, 3], [('final5kact_lstm_s0_ep5.pt', 'lstm')]),
            'final6_lstm': ('final5_act3_submission.zip', [0, 1, 2, 3, 4], [('final5kact_lstm_s0_ep5.pt', 'lstm')])}
pm3 = load_pm(B, fresh_name()); t0 = time.time(); run_pm(pm3, np.asarray(Xv[23], np.float32), 400); t_ens3 = time.time() - t0
for name, (base_zip, keep, adds) in VARIANTS.items():
    print(f'\n{name}: из {base_zip} оставляем модели {keep}, добавляем {[a for a, _ in adds]}')
    Bz = f'/content/_base_{name}'
    if os.path.exists(Bz): shutil.rmtree(Bz)
    os.makedirs(Bz); zipfile.ZipFile(f'{CKPT_DIR}/{base_zip}').extractall(Bz)
    W = fix_selftest(subset(dict(np.load(f'{Bz}/weights.npz')), keep))
    for ck, kind in adds: W = add_model(W, ck, kind)
    assert all(np.isfinite(W[k]).all() for k in ('flat', 'coef', 'const', 'selftest_y'))
    OUT = f'/content/{name}'; save_dir(W, OUT); pm = load_pm(OUT, fresh_name()); print('  самопроверка пройдена; модели (тип, …, H):', W['mods'][:, [0, 3]].tolist())
    X = np.asarray(Xv[22], np.float32); d1, d2 = run_pm(pm, X, 300), run_pm(pm, X, 301)
    print('  детерминизм:', np.array_equal(d1, d2), '| конечны:', bool(np.isfinite(d1).all()), f'| обрезано на ±2: {100 * np.mean(np.abs(d1[WARM:]) > 2):.1f}%')
    t0 = time.time(); run_pm(pm, np.asarray(Xv[23], np.float32), 401); print(f'  оценка на платформе ≈ {26.1 * (time.time() - t0) / t_ens3:.0f} мин')
    shutil.make_archive(f'/content/{name}_submission', 'zip', OUT); shutil.copy(f'/content/{name}_submission.zip', f'{CKPT_DIR}/')
    print(f'  готово: {name}_submission.zip (копия на Drive)')


final5_lstm: из final5_act3_submission.zip оставляем модели [0, 1, 2, 3], добавляем ['final5kact_lstm_s0_ep5.pt']


/tmp/ipykernel_1051/2357814792.py:93: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X_, dtype=np.float32)))[0].numpy().astype(np.float64)


  + final5kact_lstm_s0_ep5.pt (lstm): моделей 5 | расхождение движка с PyTorch на полной последовательности 1.4e-07 (нужно < 1e-3)
  самопроверка пройдена; модели (тип, …, H): [[0, 128], [1, 128], [0, 128], [0, 128], [3, 128]]
  детерминизм: True | конечны: True | обрезано на ±2: 0.0%
  оценка на платформе ≈ 43 мин
  готово: final5_lstm_submission.zip (копия на Drive)

final6_lstm: из final5_act3_submission.zip оставляем модели [0, 1, 2, 3, 4], добавляем ['final5kact_lstm_s0_ep5.pt']
  + final5kact_lstm_s0_ep5.pt (lstm): моделей 6 | расхождение движка с PyTorch на полной последовательности 2.1e-07 (нужно < 1e-3)
  самопроверка пройдена; модели (тип, …, H): [[0, 128], [1, 128], [0, 128], [0, 128], [0, 128], [3, 128]]
  детерминизм: True | конечны: True | обрезано на ±2: 0.0%
  оценка на платформе ≈ 52 мин
  готово: final6_lstm_submission.zip (копия на Drive)


In [7]:
SEED, EPF, N5 = 1, 5, 5000
evrate = lambda x: (np.diff(np.asarray(x[::4, :22], np.float32), axis=0) != 0).any(1).mean()
if 'rateT' not in globals() or len(rateT) != len(XA):
    t0 = time.time(); rateT = np.array([evrate(XA[s]) for s in range(len(XA))]); print(f'активность train: {time.time() - t0:.0f} с')
tr_ids = np.sort(np.concatenate([np.where(famT == f)[0][np.argsort(-rateT[famT == f])][:int(round(N5 * np.mean(famT == f)))] for f in range(4)]))
rngs = np.random.default_rng(200 + SEED); hold = np.sort(rngs.choice(1873, 300, replace=False))
held_idx = np.sort(np.random.default_rng(1).choice(1873, 300, replace=False))
rest_idx = np.array([i for i in range(1873) if i not in set(held_idx.tolist())])
pos = {int(i): ('v', k) for k, i in enumerate(held_idx)}; pos.update({int(i): ('r', k) for k, i in enumerate(rest_idx)})
vin = [i for i in range(1873) if i not in set(hold.tolist())]
src = [('a', int(i)) for i in tr_ids] + [pos[i] for i in vin]
ARR = {'a': (XA, YA), 'v': (Xv, Yv), 'r': (XR, YR)}
def getb(items):
    return (np.stack([np.asarray(ARR[a][0][k], np.float32) for a, k in items]), np.stack([np.asarray(ARR[a][1][k], np.float32) for a, k in items]))
famS = np.array([famT[k] if a == 'a' else (famv[k] if a == 'v' else famR[k]) for a, k in src])
N = len(src); mix = np.bincount(famS, minlength=4) / N; wseq = (VALMIX / mix)[famS]; wseq = (wseq / wseq.mean()).astype(np.float32)
hsrc = [pos[int(i)] for i in hold]

class NetL(nn.Module):                                           # LSTM 2×128 + линейная голова
    def __init__(s, H=128): super().__init__(); s.rnn = nn.LSTM(112, H, 2, batch_first=True); s.head = nn.Linear(H, 2)
    def forward(s, x, h=None): y, h = s.rnn(x, h); return s.head(y), h
@torch.no_grad()
def hold_score(net):
    net.eval(); S = np.zeros((2, 6))
    for s in range(0, 300, 50):
        xb, yb = getb(hsrc[s:s + 50]); x = torch.from_numpy(xb).to(dev); h = None; o_ = []
        for t in range(0, L, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
        S += sums(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy(), yb, np.broadcast_to(post, (len(xb), L)))
    net.train(); return corrT(S)
print(f'LSTM, сид {SEED}: train {len(tr_ids)} самых активных + valid {len(vin)} | отложено 300 valid')
torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
net = NetL().to(dev); opt = torch.optim.Adam(net.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPF * (N // 32) * (L // 1000), pct_start=0.05)
for ep in range(EPF):
    net.train(); t0 = time.time(); order = rng.permutation(N); run = []; skipped = 0
    for g in range(N // 32):
        k = order[g * 32:(g + 1) * 32]; xb, yb = getb([src[i] for i in k])
        x = torch.from_numpy(xb).to(dev); y = torch.from_numpy(yb).to(dev).clamp(-2, 2); ws = torch.from_numpy(wseq[k]).to(dev); h = None
        for t in range(0, L, 1000):
            o, h = net(x[:, t:t + 1000], h); h = tuple(v.detach() for v in h); yy = y[:, t:t + 1000]
            l = ws[:, None, None] * yy.abs() * (o - yy) ** 2
            if t < WARM: l = l[:, WARM - t:]
            l = l.mean()
            if not torch.isfinite(l): opt.zero_grad(set_to_none=True); sched.step(); skipped += 1; h = None; continue
            opt.zero_grad(set_to_none=True); l.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step(); run.append(l.item())
    assert all(torch.isfinite(p).all() for p in net.parameters()), 'веса испорчены — бери прошлую эпоху'
    torch.save(net.state_dict(), f'{CKPT_DIR}/final5kact_lstm_s{SEED}_ep{ep + 1}.pt')
    print(f'эпоха {ep + 1}/{EPF} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | пропущено {skipped} | '
          f'отложенные 300 valid, все строки {hold_score(net):.4f}', flush=True)

LSTM, сид 1: train 5000 самых активных + valid 1573 | отложено 300 valid
эпоха 1/5 | 347 с | loss 0.8486 | пропущено 0 | отложенные 300 valid, все строки 0.4789
эпоха 2/5 | 347 с | loss 0.8026 | пропущено 0 | отложенные 300 valid, все строки 0.4877
эпоха 3/5 | 345 с | loss 0.7885 | пропущено 0 | отложенные 300 valid, все строки 0.4880
эпоха 4/5 | 346 с | loss 0.7778 | пропущено 0 | отложенные 300 valid, все строки 0.4917
эпоха 5/5 | 343 с | loss 0.7688 | пропущено 0 | отложенные 300 valid, все строки 0.4919


In [8]:
# ===== Сборка final5_2lstm = 2 модели Андрея + активная GRU s2 + LSTM s0 + LSTM s1 =====
for f in ('final5_act3_submission.zip', 'final5kact_lstm_s0_ep5.pt', 'final5kact_lstm_s1_ep5.pt'):
    assert os.path.exists(f'{CKPT_DIR}/{f}'), f'нет {f} на Drive — дождись, пока доучится LSTM с сидом 1'
for fn in ('patch_lstm', 'subset', 'fix_selftest', 'add_model', 'save_dir', 'load_pm', 'run_pm', 'NetL'):
    assert fn in globals(), f'нет {fn}: среда сбросилась — запусти целиком ячейку сборки с LSTM, заменив в ней VARIANTS на строку из этой ячейки'

VARIANTS = {'final5_2lstm': ('final5_act3_submission.zip', [0, 1, 2],
                             [('final5kact_lstm_s0_ep5.pt', 'lstm'), ('final5kact_lstm_s1_ep5.pt', 'lstm')])}
pm3 = load_pm(B, fresh_name()); t0 = time.time(); run_pm(pm3, np.asarray(Xv[23], np.float32), 400); t_ens3 = time.time() - t0
for name, (base_zip, keep, adds) in VARIANTS.items():
    print(f'{name}: из {base_zip} оставляем модели {keep}, добавляем {[a for a, _ in adds]}')
    Bz = f'/content/_base_{name}'
    if os.path.exists(Bz): shutil.rmtree(Bz)
    os.makedirs(Bz); zipfile.ZipFile(f'{CKPT_DIR}/{base_zip}').extractall(Bz)
    W = fix_selftest(subset(dict(np.load(f'{Bz}/weights.npz')), keep))
    for ck, kind in adds: W = add_model(W, ck, kind)
    assert all(np.isfinite(W[k]).all() for k in ('flat', 'coef', 'const', 'selftest_y'))
    OUT = f'/content/{name}'; save_dir(W, OUT); pm = load_pm(OUT, fresh_name()); print('  самопроверка пройдена; модели (тип, H):', W['mods'][:, [0, 3]].tolist())
    X = np.asarray(Xv[22], np.float32); d1, d2 = run_pm(pm, X, 300), run_pm(pm, X, 301)
    print('  детерминизм:', np.array_equal(d1, d2), '| конечны:', bool(np.isfinite(d1).all()), f'| обрезано на ±2: {100 * np.mean(np.abs(d1[WARM:]) > 2):.1f}%')
    t0 = time.time(); run_pm(pm, np.asarray(Xv[23], np.float32), 401); print(f'  оценка на платформе ≈ {26.1 * (time.time() - t0) / t_ens3:.0f} мин')
    shutil.make_archive(f'/content/{name}_submission', 'zip', OUT); shutil.copy(f'/content/{name}_submission.zip', f'{CKPT_DIR}/')
    print(f'  готово: {name}_submission.zip (копия на Drive)')

final5_2lstm: из final5_act3_submission.zip оставляем модели [0, 1, 2], добавляем ['final5kact_lstm_s0_ep5.pt', 'final5kact_lstm_s1_ep5.pt']
  + final5kact_lstm_s0_ep5.pt (lstm): моделей 4 | расхождение движка с PyTorch на полной последовательности 1.5e-07 (нужно < 1e-3)
  + final5kact_lstm_s1_ep5.pt (lstm): моделей 5 | расхождение движка с PyTorch на полной последовательности 1.3e-07 (нужно < 1e-3)
  самопроверка пройдена; модели (тип, H): [[0, 128], [1, 128], [0, 128], [3, 128], [3, 128]]
  детерминизм: True | конечны: True | обрезано на ±2: 0.0%
  оценка на платформе ≈ 43 мин
  готово: final5_2lstm_submission.zip (копия на Drive)
